In [1826]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy

In [1827]:
np.random.seed(0)

In [1828]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [1830]:
def manipulation_thresholds(thresholds, priors, c):
    if priors.sum() == 0:
        return thresholds, priors, thresholds
    manip_thresholds = np.maximum(0, thresholds - (bayesian_update(priors) / c))
    return manip_thresholds

In [1831]:
def merge_classifiers(thresholds, priors, manip_thresholds):
    if len(thresholds) <= 1:
        return thresholds, priors, manip_thresholds
    
    merged_thresholds = [thresholds[0]]
    merged_priors = [priors[0]]
    merged_manip_thresholds = [manip_thresholds[0]]

    for i in range(1, len(thresholds)):
        if manip_thresholds[i] <= merged_manip_thresholds[-1]:
            merged_thresholds[-1] = thresholds[i]
            merged_priors[-1] += priors[i]
            merged_manip_thresholds[-1] = manip_thresholds[i]
        else:
            merged_thresholds.append(thresholds[i])
            merged_priors.append(priors[i])
            merged_manip_thresholds.append(manip_thresholds[i])
    
    return np.array(merged_thresholds), np.array(merged_priors), np.array(merged_manip_thresholds)

In [1832]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [1833]:
def accuracy_loss(thresholds, priors, manip_thresholds, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        if thresholds[i] < threshold_true:
            loss = (threshold_true - manip_thresholds[i])
        else:
            loss = np.abs(manip_thresholds[i] - threshold_true)
        losses.append(loss)
    losses = np.array(losses)
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [1834]:
def evaluate_partition(partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    manip_thresholds_p = manipulation_thresholds(thresholds_p, priors_p, c)
    thresholds_p, priors_p, manip_thresholds_p = merge_classifiers(thresholds_p, priors_p, manip_thresholds_p)
    acc_loss_p = accuracy_loss(thresholds_p, priors_p, manip_thresholds_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p, manip_thresholds_p
    return acc_loss_p

def evaluate_system(partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [1835]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [1836]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)


def find_partitions_greedy(thresholds, priors, threshold_true, c):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1

    Q = collections.deque(itertools.combinations(P.keys(), 2))
    while Q:
        # display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    Q.append((new_id, p_id))
    return list(P.values())

In [1837]:
def find_partitions_optimal(thresholds, priors, threshold_true, c):
    indices = [i for i in range(len(thresholds))]

    parts = set_partitions(indices)
    partitions_set = []
    for part in parts:
        partitions_set.append(part)

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.
        for partition in partitions:
            acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
            acc_loss += acc_loss_p * np.sum(priors[partition])

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [1838]:
c = 5.0
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

# priors = np.zeros_like(thresholds)
# balance_priors(priors, random=True)
priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])

print(np.sum(priors))
pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.0


,0,1,2,3,4,5,6,7,8
threshold,0.100,0.200,0.300,0.40,0.500,0.600,0.700,0.800,0.90
priors,0.114,0.115,0.036,0.25,0.022,0.054,0.062,0.097,0.25


In [1839]:
partition_greedy = find_partitions_greedy(thresholds, priors, threshold_true, c)
partition_optimal = find_partitions_optimal(thresholds, priors, threshold_true, c)

acc_loss_greedy = evaluate_system(partition_greedy, thresholds, priors, threshold_true, c)
acc_loss_optimal = evaluate_system(partition_optimal, thresholds, priors, threshold_true, c)

In [1840]:
print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal)}")
print(f"Acc Loss : {acc_loss_optimal:.4f}")

Greedy
------
Partition: [[0, 1, 2, 3, 4, 5], [6], [7], [8]]
Acc Loss : 0.2069

Optimal
-------
Partition: [[0, 1, 2, 3, 4, 5], [6], [7], [8]]
Acc Loss : 0.2069


In [1841]:
def find_mismatches(n, tol=0.):
    for _ in tqdm.trange(1000):
        thresholds = np.sort(np.random.rand(n))
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        if abs(1-np.sum(priors)) > 1e-3:
            print(np.sum(priors))
            continue
        
        threshold_true = np.random.rand()
        c = np.random.uniform(0.05, 0.5)

        partition_opt = find_partitions_optimal(thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(thresholds, priors, threshold_true, c)

        acc_loss_opt = evaluate_system(partition_opt, thresholds, priors, threshold_true, c)
        acc_loss_greedy = evaluate_system(partition_greedy, thresholds, priors, threshold_true, c)

        if (1 - acc_loss_opt)/(1- acc_loss_greedy) > tol:
        # if acc_loss_greedy - acc_loss_opt > tol:
            return thresholds, priors, threshold_true, c

In [1842]:
thresholds_ce, priors_ce, threshold_true_ce, c_ce = find_mismatches(10, tol=1.009)

  0%|          | 2/1000 [00:20<2:50:40, 10.26s/it]


In [1843]:
# c_ce = 10.

partition_optimal_ce = find_partitions_optimal(thresholds_ce, priors_ce, threshold_true_ce, c_ce)
partition_greedy_ce = find_partitions_greedy(thresholds_ce, priors_ce, threshold_true_ce, c_ce)

acc_loss_greedy_ce = evaluate_system(partition_greedy_ce, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
acc_loss_optimal_ce = evaluate_system(partition_optimal_ce, thresholds_ce, priors_ce, threshold_true_ce, c_ce)

In [1844]:
print(f"t*        : {threshold_true_ce:.4f}")
print(f"c         : {c_ce:.4f}")
print(f"Thresholds: {thresholds_ce.round(4)}")
print(f"Priors    : {priors_ce.round(4)}")
print()

print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy_ce)}")
print(f"Acc Loss : {acc_loss_greedy_ce:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal_ce)}")
print(f"Acc Loss : {acc_loss_optimal_ce:.4f}")
print()
print(f"Ratio: {(1 - acc_loss_optimal_ce)/(1- acc_loss_greedy_ce):.4f}")

t*        : 0.1966
c         : 0.2159
Thresholds: [0.102  0.1289 0.2104 0.3154 0.3637 0.4386 0.5702 0.6668 0.6706 0.9884]
Priors    : [0.0685 0.0529 0.2141 0.083  0.1528 0.0801 0.0521 0.0362 0.2151 0.0453]

Greedy
------
Partition: [[0, 1, 2, 3, 4, 6, 7], [5, 8, 9]]
Acc Loss : 0.1865

Optimal
-------
Partition: [[0, 4, 9], [1, 2, 5, 6, 7, 8], [3]]
Acc Loss : 0.1778

Ratio: 1.0107


In [1845]:
a = [0,4]
b = [9]

acc_loss_a, thresholds_a, priors_a, manip_thresholds_a = evaluate_partition(a, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
acc_loss_b, thresholds_b, priors_b, manip_thresholds_b = evaluate_partition(b, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

lhs = acc_loss_a * np.sum(priors_ce[a]) + acc_loss_b * np.sum(priors_ce[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab, manip_thresholds_ab = evaluate_partition(ab, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
rhs = acc_loss_ab * np.sum(priors_ce[ab])

acc_loss_a, acc_loss_b, acc_loss_ab, lhs-rhs
manip_thresholds_a, manip_thresholds_b, manip_thresholds_ab, thresholds_ab

(array([0.]),
 array([0.]),
 array([0.        , 0.20158089]),
 array([0.36371077, 0.98837384]))

In [1846]:
import time

def benchmark_runtime(n):
    times_opt = []
    times_greedy = []
    for _ in tqdm.trange(10):
        thresholds = np.sort(np.random.rand(n))
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        if abs(1-np.sum(priors)) > 1e-3:
            print(np.sum(priors))
            break
        
        threshold_true = np.random.rand()
        c = np.random.uniform(0.05, 0.5)

        start_opt = time.perf_counter()
        find_partitions_optimal(thresholds, priors, threshold_true, c)
        end_opt = time.perf_counter()
        
        start_greedy = time.perf_counter()
        find_partitions_greedy(thresholds, priors, threshold_true, c)
        end_greedy = time.perf_counter()

        times_opt.append(end_opt - start_opt)
        times_greedy.append(end_greedy - start_greedy)

    return np.mean(times_opt), np.mean(times_greedy)

In [1847]:
times_opt = []
times_greedy = []
for i in range(1,11):
    time_opt, time_greedy = benchmark_runtime(i)
    times_opt.append(time_opt)
    times_greedy.append(time_greedy)

100%|██████████| 10/10 [01:17<00:00,  7.80s/it]


In [1848]:
results = {"n": [i for i in range(1,11)], "opt": times_opt, "greedy": times_greedy}
px.scatter(results, x="n", y=["opt", "greedy"], width=800, height=600, title="Run Time Performance (s)").update_layout(margin=dict(t=50,b=25,l=25,r=25), title=dict(x=0.5))